# ممیزی متادیتای Qwen3.5-2B برای پروژهٔ Mobile MoE

هدف پروژه: مدل نهایی با حدود ۲ میلیارد پارامتر یکتا، مسیر اجرا آفلاین روی اندروید و تمرکز بر فارسی و استدلال.

این Notebook نسخهٔ دقیق config را با SHA ثبت می‌کند و اطلاعات tokenizer و Hub را می‌خواند. سلول شمارش، فقط headerهای فایل‌های safetensors را با HTTP Range می‌خواند؛ **محتوای وزن‌ها دانلود نمی‌شود و هیچ آموزش یا سرویس پولی اجرا نمی‌شود.** شمارش عناصر tensor با شمارش دقیق پارامترهای قابل‌آموزش یکی فرض نمی‌شود.

In [ ]:
import json, urllib.request, urllib.error, datetime
MODEL_ID = 'Qwen/Qwen3.5-2B'
REVISION = 'main'
HEADERS = {'User-Agent': 'Qwen-MoE-config-audit/0.2'}
def get_json(url):
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=45) as response:
        return json.load(response)
api_url = f'https://huggingface.co/api/models/{MODEL_ID}/revision/{REVISION}?blobs=true'
info = get_json(api_url)
SHA = info.get('sha')
assert SHA, 'Hub did not return a resolved commit SHA'
BASE = f'https://huggingface.co/{MODEL_ID}/resolve/{SHA}'
config = get_json(f'{BASE}/config.json')
try:
    tokenizer_config = get_json(f'{BASE}/tokenizer_config.json')
except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, json.JSONDecodeError):
    tokenizer_config = {}
print('Model:', MODEL_ID)
print('Resolved revision:', SHA)
print('License:', info.get('license') or info.get('cardData', {}).get('license'))
print('Architecture:', config.get('architectures'))
print('Model type:', config.get('model_type'))


In [ ]:
text = config.get('text_config', config)
hidden = int(text.get('hidden_size', 0) or 0)
vocab = int(text.get('vocab_size', config.get('vocab_size', 0)) or 0)
intermediate = int(text.get('intermediate_size', 0) or 0)
layers = int(text.get('num_hidden_layers', 0) or 0)
layer_types = text.get('layer_types', [])
print('Hidden size:', hidden)
print('Vocabulary:', vocab)
print('Layers:', layers)
print('Dense FFN intermediate size:', intermediate)
print('Layer type counts:', {name: layer_types.count(name) for name in sorted(set(layer_types))})
print('Tied embeddings:', text.get('tie_word_embeddings', config.get('tie_word_embeddings')))
print('Embedding matrix formula:', f'{hidden*vocab:,}', f'({hidden*vocab/1e6:.1f}M parameters)')
print('Tokenizer class:', tokenizer_config.get('tokenizer_class'))
print('Chat template present:', bool(tokenizer_config.get('chat_template')))
print('Hub safetensors parameter metadata:', info.get('safetensors', {}).get('parameters'))
print('Safetensors files:', [f.get('rfilename') for f in info.get('siblings', []) if f.get('rfilename', '').endswith('.safetensors')])


## شمارش عناصر ذخیره‌شده با خواندن header (بدون دانلود وزن‌ها)

Safetensors فایل‌های وزن را با یک header شامل نام و shape تانسورها آغاز می‌کند. این سلول فقط چند بایت اول و سپس header را با درخواست HTTP Range می‌خواند. اگر میزبان Range را پشتیبانی نکند یا درخواست خطا بدهد، خطا را گزارش می‌کند و نباید آن را به معنی شمارش صفر تفسیر کرد.

In [ ]:
import math, struct
def read_safetensors_header(url, max_header_bytes=32 * 1024 * 1024):
    def ranged(start, end):
        req = urllib.request.Request(url, headers={**HEADERS, 'Range': f'bytes={start}-{end}'})
        with urllib.request.urlopen(req, timeout=60) as response:
            return response.read(end - start + 1)
    first = ranged(0, 7)
    if len(first) != 8:
        raise RuntimeError('Could not read 8-byte safetensors header length')
    header_size = struct.unpack('<Q', first)[0]
    if header_size <= 0 or header_size > max_header_bytes:
        raise RuntimeError(f'Unreasonable header size: {header_size}')
    raw = ranged(8, 7 + header_size)
    if len(raw) != header_size:
        raise RuntimeError(f'Truncated header: expected {header_size}, got {len(raw)}')
    header = json.loads(raw.decode('utf-8'))
    tensors = [(name, meta) for name, meta in header.items() if name != '__metadata__' and isinstance(meta, dict)]
    return header_size, tensors
header_results, header_errors = [], []
for file in info.get('siblings', []):
    filename = file.get('rfilename', '')
    if not filename.endswith('.safetensors'):
        continue
    try:
        header_size, tensors = read_safetensors_header(f'{BASE}/{filename}')
        numel = sum(math.prod(meta.get('shape', [])) if meta.get('shape') else 1 for _, meta in tensors)
        header_results.append({'file': filename, 'header_bytes_read': header_size, 'tensor_count': len(tensors), 'serialized_tensor_elements': numel})
    except Exception as exc:
        header_errors.append({'file': filename, 'error': str(exc)})
print('Header audit:', json.dumps(header_results, ensure_ascii=False, indent=2))
print('Header errors:', json.dumps(header_errors, ensure_ascii=False, indent=2))
print('Total serialized tensor elements:', sum(item['serialized_tensor_elements'] for item in header_results))
print('Caveat: serialized tensor elements can include buffers and are not automatically equal to trainable parameters.')


## برآورد اولیهٔ MoE (فرمولی، نه شمارش checkpoint)

این بخش فقط هزینهٔ FFNهای متخصص را با فرض SwiGLU سه‌ماتریسی و بدون bias محاسبه می‌کند. attention، Router، embedding/LM head، normها و سایر اجزا در آن نیستند؛ بنابراین active FFN را نباید «کل پارامتر فعال مدل» نامید.

In [ ]:
NUM_EXPERTS = 8
TOP_K = 2
EXPERT_INTERMEDIATE = 768
params_per_expert = 3 * hidden * EXPERT_INTERMEDIATE
total_expert_ffn = params_per_expert * NUM_EXPERTS * layers
active_expert_ffn = params_per_expert * TOP_K * layers
dense_ffn_estimate = 3 * hidden * intermediate * layers
report = {
    'model_id': MODEL_ID,
    'resolved_revision': SHA,
    'audited_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'architecture': config.get('architectures'),
    'text_config': {'hidden_size': hidden, 'vocab_size': vocab, 'num_hidden_layers': layers,
                    'intermediate_size': intermediate, 'layer_types': layer_types},
    'hub_parameter_metadata': info.get('safetensors', {}).get('parameters'),
    'formula_estimates_not_checkpoint_counts': {
        'embedding_matrix_parameters': hidden * vocab,
        'dense_swiglu_ffn_all_layers': dense_ffn_estimate,
        'experts_per_converted_layer': NUM_EXPERTS, 'top_k': TOP_K,
        'expert_intermediate_size': EXPERT_INTERMEDIATE,
        'parameters_per_expert_ffn': params_per_expert,
        'all_expert_ffn_parameters': total_expert_ffn,
        'active_expert_ffn_parameters_per_token': active_expert_ffn,
    },
    'scope': 'Config and Hub metadata only; no weight tensors downloaded; no training performed.'
}
print(json.dumps(report, ensure_ascii=False, indent=2))
with open('qwen35_moe_audit_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print('Saved qwen35_moe_audit_report.json')


## تفسیر و گام بعدی

۱. SHA و متادیتای Hub را ثبت می‌کنیم تا ممیزی قابل‌بازتولید باشد.
۲. اگر Hub تعداد پارامترها را اعلام نکند، گزارش آن را خالی می‌گذارد؛ از اندازهٔ فایل به‌تنهایی تعداد پارامتر دقیق نتیجه نمی‌گیریم.
۳. شمارش دقیق checkpoint باید با خواندن metadata/header فایل‌های safetensors یا بارگذاری وزن‌ها در محیط مناسب انجام شود. پیش از هر دانلود، فضای دیسک و RAM محیط بررسی می‌شود.
۴. پس از ممیزی، اندازهٔ expertها را طوری تنظیم می‌کنیم که مجموع پارامترهای یکتا نزدیک ۲B بماند.
۵. اولین پیاده‌سازی فقط یک FFN-MoE آزمایشی خواهد بود؛ آموزش گسترده و محاسبات پولی بدون اجازه اجرا نمی‌شود.